# Practice 34 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — New Concept: Vectorization vs `.apply()`

### What is vectorization?

**`.apply()`** loops over rows or values in Python — one at a time. Python loops are slow because of interpreter overhead.

**Vectorized operations** work on the entire array at once using optimized C code under the hood — no Python loop, much faster.

```python
# Slow — Python loop element by element
df['tip_pct'] = df.apply(lambda row: row['tip'] / row['total_bill'], axis=1)

# Fast — vectorized, operates on whole columns at once
df['tip_pct'] = df['tip'] / df['total_bill']
```

Common vectorized alternatives to `.apply()`:

| Instead of `.apply()` doing... | Use... |
|---|---|
| arithmetic between columns | direct operators: `+`, `-`, `/`, `*` |
| simple if/else on one column | `np.where(condition, true_val, false_val)` |
| string operations | `.str.upper()`, `.str.contains()`, `.str.replace()` |
| binning values | `pd.cut()` or `pd.qcut()` |
| math functions | `np.log()`, `np.sqrt()`, `np.abs()` |

**When `.apply()` is still the right choice:**
- Logic that depends on multiple columns with complex conditions (hard to express with `np.where`)
- Custom functions pandas can't vectorize
- Small DataFrames where readability matters more than speed

### Measuring performance with `%timeit`

```python
%timeit df['tip'] / df['total_bill']                              # vectorized
%timeit df.apply(lambda row: row['tip'] / row['total_bill'], axis=1)  # apply
```

`%timeit` runs the expression many times and reports the average. The difference is often 10–100x.

---
### Task 1: tips — compare approaches

Load `sns.load_dataset('tips')`.

1. Add `tip_pct` two ways: once with `.apply()`, once vectorized. Use `%timeit` on both. How much faster is the vectorized version?
2. Add a `size_label` column: `'large'` if `size >= 4`, else `'small'`. Do it with `.apply()` first, then rewrite it using `np.where`.
3. Add a `bill_category` column using `pd.cut` with 3 equal-width bins, labeled `'low'`, `'mid'`, `'high'`. Could you do this with `.apply()`? Why is `pd.cut` better here?
4. Convert the `sex` column to uppercase using `.str.upper()`. Write the equivalent `.apply()` version — notice how much more verbose it is.

In [ ]:
# Your code here
tips = sns.load_dataset('tips')
%timeit tips['tip_pct'] = tips.apply(lambda row: row['tip']/row['total_bill'], axis = 1)
%timeit tips['tip_pct'] = tips['tip']/tips['total_bill']
### vectorized is about 20 times faster 




4.16 ms ± 552 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
248 µs ± 36.1 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


21.428571428571427

In [8]:
tips['size_label'] =tips['size'].apply(lambda x: 'larger' if x>=4
                                       else 'small')
tips['size_label'] = np.where(tips['size']>= 4, 'large','small')

In [ ]:
tips['bill_category'] = pd.cut(tips['total_bill'], 3, labels=['low','mid','high'])

ss = np.percentile(tips['total_bill'], [33,66])
tips['bill_category'] = tips['total_bill'].apply(lambda x: 'high' if x>ss[1]
                                                 else 'mid' if x >ss[0]
                                                 else 'low')

###pd.cut is better because you don't need to calulate the quantiles yourself

In [17]:
tips['sex'] = tips['sex'].str.upper()
tips['sex'] = tips['sex'].apply(lambda x: x.upper())

---
## Level 2 — Rewrite Challenge

### Task 2: rewrite these `.apply()` calls

Each snippet below uses `.apply()`. Rewrite each one as a vectorized operation — no `.apply()` allowed.

```python
diamonds = sns.load_dataset('diamonds')

# A — arithmetic
diamonds['ppc'] = diamonds.apply(lambda row: row['price'] / row['carat'], axis=1)

# B — simple if/else
diamonds['expensive'] = diamonds['price'].apply(lambda x: True if x > 5000 else False)

# C — math function
diamonds['log_price'] = diamonds['price'].apply(lambda x: np.log(x))

# D — string operation
diamonds['cut_upper'] = diamonds['cut'].apply(lambda x: str(x).upper())

# E — multi-condition if/else (3 tiers)
diamonds['tier'] = diamonds['price'].apply(
    lambda x: 'premium' if x > 10000 else 'mid' if x > 3000 else 'budget'
)
```

After rewriting all five, use `%timeit` to compare your vectorized version vs the original `.apply()` on snippet **A** and **E**.

In [25]:
diamonds = sns.load_dataset('diamonds')

# Your rewrites here

#A
diamonds['ppc'] = diamonds['price']/diamonds['carat']
#B
diamonds['expensive'] = diamonds['price']>5000
#C
diamonds['log_price'] = np.log(diamonds['price'])
#D
diamonds['cut_upper'] = diamonds['cut'].str.upper()
#E 
diamonds['tier'] = pd.cut(diamonds['price'], 
                          bins=[0,3000,10000,np.inf], labels=['budget', 'mid','premium'])



In [26]:
# compare 
%timeit diamonds['ppc'] = diamonds.apply(lambda row: row['price'] / row['carat'], axis=1)
%timeit diamonds['ppc'] = diamonds['price']/diamonds['carat']

%timeit diamonds['tier'] = diamonds['price'].apply(lambda x: 'premium' if x > 10000 else 'mid' if x > 3000 else 'budget')

%timeit diamonds['tier'] = pd.cut(diamonds['price'], bins=[0,3000,10000,np.inf], labels=['budget', 'mid','premium'])

474 ms ± 29.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
258 µs ± 16.7 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
9.95 ms ± 117 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
1.19 ms ± 30.8 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


---
## Level 3 — When to use which

### Task 3: penguins — choose the right tool

Load `sns.load_dataset('penguins')`. Drop nulls.

For each of the following, decide whether to use vectorization or `.apply()` — and justify your choice in a comment:

1. Add `bill_ratio` (`bill_length_mm / bill_depth_mm`).
2. Add `size_class`: `'large'` if `body_mass_g > 4500` else `'small'`.
3. Add `island_species` — a string combining island and species: e.g. `'Torgersen_Adelie'`.
4. Add `mass_class` — `'heavy'` if `body_mass_g > species mean` AND `flipper_length_mm > species mean`, `'light'` if both are below, `'mixed'` otherwise. (Hint: think carefully about whether this can be vectorized.)

After adding all columns:
- Group by `species` and `mass_class` — what's the mean `bill_ratio` per group? Use `observed=True`.
- Which species has the most `'heavy'` penguins? Use `np.argmax()`.

In [59]:
# Your code here
penguins = sns.load_dataset('penguins').dropna()

penguins['bill_ratio'] = penguins['bill_length_mm']/penguins['bill_depth_mm']
# vecterization is simple and fast 

penguins['size_class'] = pd.cut(penguins['body_mass_g'], bins =[0, 4500, np.inf], labels=['small','large'])
# vecterization is simple and fast 

penguins['island_species'] = penguins['island']+'_'+penguins['species']
#penguins['island_species'] = penguins.apply(lambda row: row['island']+'_'+ row['species'], axis = 1)
# vecterization is simple and fast 
penguins['species_mean_mass'] = penguins.groupby('species', observed='True')['body_mass_g'].transform('mean')
penguins['species_mean_fl'] = penguins.groupby('species', observed='True')['flipper_length_mm'].transform('mean')

penguins['mass_class'] = penguins.apply(lambda row: 'heavy' if row['body_mass_g']>row['species_mean_mass'] and row['flipper_length_mm']>row['species_mean_fl']
                                        else 'light' if row['body_mass_g']<row['species_mean_mass'] and row['flipper_length_mm']<row['species_mean_fl']
                                         else 'mixed', axis = 1)
## maybe can be vectorized using nested np.where?

penguins.groupby(['species','mass_class'], observed= True)['bill_ratio'].mean()

m = penguins.groupby('species')['body_mass_g'].max()
m.index[np.argmax(m)]


m = penguins[penguins['mass_class']=="heavy"].groupby('species',observed = True).size()
m.index[np.argmax(m)]


'Adelie'

---
## Level 4 — Real-world

### Task 4: mpg — pipeline with vectorization

Load `sns.load_dataset('mpg')`. Drop rows where `horsepower` is null.

Write a `.pipe()` chain with 3 functions. This time, **avoid `.apply()` wherever possible** — use vectorized operations throughout:

- One converts `cylinders` (ordered) and `origin` (unordered) to `category`
- One adds `power_to_weight` and `log_weight` (`np.log`) — both vectorized
- One adds an `efficiency` label — `'high'` (mpg > 30), `'mid'` (20–30), `'low'` (< 20) — use `pd.cut` or `np.where`, not `.apply()`

After the chain:
- Which origin has the highest mean `power_to_weight`? Use `np.argmax()`.
- Build a pivot table: mean `log_weight` by `efficiency` × `origin`, `observed=True`.
- Use `.loc` to extract only `'high'` efficiency cars. What fraction are from Japan?

In [56]:
# Your code here
mpg = sns.load_dataset('mpg').dropna(subset='horsepower')

def cv(df):
    co = pd.CategoricalDtype(categories=[3,4,5,6,8], ordered=True)
    df['cylinders'] = df['cylinders'].astype(co)
    df['origin'] = df['origin'].astype('category')
    return df 

def adp(df):
    df['power_to_weight'] = df['horsepower']/df['weight']
    df['log_weight'] = np.log(df['weight'])
    return df 

def ade(df):
    df['efficiency'] = pd.cut(df['mpg'], bins=[0,20,30,np.inf], labels=['low','mid','high'])
    return df 

mpg = mpg.pipe(cv).pipe(adp).pipe(ade)

m = mpg.groupby('origin', observed=True)['power_to_weight'].mean()
m.index[np.argmax(m)]


p = mpg.pivot_table(
    index = 'efficiency',
    columns = 'origin',
    values = 'log_weight',
    observed = True
)

(mpg.loc[mpg['efficiency']== 'high','origin']=='japan').mean()



np.float64(0.5542168674698795)